In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    Safely retrieve dataName from sys.argv.
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
dataName = "Variables"

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracked_Profiles", dataName="Tracked_2dHistogram",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
##############################################
#JOB ARRAY

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=False

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary
    
def GetSpatialData(t):    
    variableNames = ['Z']
    dataDictionary = MakeDataDictionary(variableNames,t)
    [Z] = (dataDictionary[k] for k in variableNames)
    return Z

In [ ]:
####################################
#RUN SETUP

In [ ]:
#data variable list
def GetVarNames(dataName): 
    Processed_string = "PROCESSED_" if "PROCESSED_" in dataName else ""
    DivideMassFlux_string = "_DivideMassFlux" if "_DivideMassFlux" in dataName else ""

    
    if dataName=="Variables":
        varNames = ['W', 'QV','QCQI', 'RH_vapor', 'THETA_v', 'THETA_e', 'MSE', 'HMC', 'CONVERGENCE','VMF_g','VMF_c']
    elif dataName == "Entrainment":
        varNames = ['D_c', 'D_g', 'E_c', 'E_g',
                          'TransferD_c', 'TransferD_g', 
                          'TransferE_c', 'TransferE_g']
    elif dataName == f"{Processed_string}Entrainment{DivideMassFlux_string}":
        varNames = [f'{Processed_string}D{DivideMassFlux_string}_c', f'{Processed_string}D{DivideMassFlux_string}_g', 
                    f'{Processed_string}E{DivideMassFlux_string}_c', f'{Processed_string}E{DivideMassFlux_string}_g',
                    f'{Processed_string}TransferD{DivideMassFlux_string}_c', f'{Processed_string}TransferD{DivideMassFlux_string}_g', 
                    f'{Processed_string}TransferE{DivideMassFlux_string}_c', f'{Processed_string}TransferE{DivideMassFlux_string}_g']
    elif dataName=="W_Budgets":
        varNames = ['WB_BUOY', 'WB_HADV', 'WB_HIDIFF', 'WB_HTURB',
                    'WB_PGRAD', 'WB_VADV', 'WB_VIDIFF', 'WB_VTURB']
    elif dataName=="QV_Budgets":
        varNames = ['QVB_HADV', 'QVB_HIDIFF', 'QVB_HTURB', 'QVB_MP',
                    'QVB_VADV', 'QVB_VIDIFF', 'QVB_VTURB']
    elif dataName=="TH_Budgets":
        varNames = ['PTB_DISS', 'PTB_DIV', 'PTB_HADV', 'PTB_HIDIFF', 
                    'PTB_HTURB', 'PTB_MP', 'PTB_RAD', 'PTB_VADV', 
                    'PTB_VIDIFF', 'PTB_VTURB']
    return varNames

In [ ]:
########################################
#RUNNING FUNCTIONS

In [ ]:
#Functions for Initializing Profile Arrays
def CopyStructure(dictionary, placeholder=None):
    """Deep-copy dictionary structure, replacing leaves with a given placeholder."""
    if isinstance(dictionary, dict):
        return {k: CopyStructure(v, placeholder) for k, v in dictionary.items()}
    else:
        return placeholder

def InitializeProfileArrays(trackedArrays, varNames, zhs=ModelData.zh):
    """
    Create a new dictionary with the same nested structure as trackedArrays,
    and for each variable name, create:
        - 'profile_array' / 'profile_array_squares'
        - 'profile_array_left' / 'profile_array_left_squares'
        - 'profile_array_right' / 'profile_array_right_squares'
    Each array has shape (len(zhs), 3) and zhs in the last column.
    """
    profileArraysDictionary = {}

    for category, depth_dict in trackedArrays.items():  # e.g. 'CL', 'SBF'
        profileArraysDictionary[category] = {}

        for depth_type in depth_dict.keys():  # e.g. 'ALL', 'SHALLOW', 'DEEP'
            profileArraysDictionary[category][depth_type] = {}

            for varName in varNames:
                # Create base profile array
                base_profile = np.zeros((len(zhs), num_bins), dtype=np.float32) #*2dHistogram
                base_count = np.zeros((len(zhs), num_bins), dtype=np.int32) #*2dHistogram

                profileArraysDictionary[category][depth_type][varName] = {
                    # Main / all parcels
                    "profile_array": base_profile.copy(),
                    "profile_array_count": base_count.copy(),

                    # Left subset (-1)
                    "profile_array_left": base_profile.copy(),
                    "profile_array_left_count": base_count.copy(),

                    # Right subset (+1)
                    "profile_array_right": base_profile.copy(),
                    "profile_array_right_count": base_count.copy(),
                }
    return profileArraysDictionary

In [ ]:
def GetParcelNumbers(trackedArray, t):
    """
    Return all parcel indices (p) and their corresponding row indices
    for parcels that are active at time t.
    Vectorized, no row-by-row loops.
    """
    t_start = trackedArray[:, 1]
    t_end   = np.minimum(trackedArray[:, 2] + trackedArray[:, 3], ModelData.Ntime)

    # Boolean mask for rows active at time t
    mask = (t >= t_start) & (t <= t_end)

    # Extract parcel numbers and their corresponding row indices
    selectedRows = np.where(mask)[0]
    selectedPs = trackedArray[selectedRows, 0]
    leftRightIndexes = trackedArray[selectedRows, 4]

    return selectedRows, selectedPs, leftRightIndexes

In [ ]:
def MakeTrackedProfiles_2dHistogram(trackedArrays, profileArraysDictionary, varNames, VARs, Z, t, variableBinsDictionary): #*2dHistogram
    # Define vertical bin edges based on model levels (Z)
    z_bins = np.arange(len(ModelData.zh) + 1)
    
    for key1, subdict in trackedArrays.items():
        for key2, trackedArray in subdict.items():
            profileArray = profileArraysDictionary[key1][key2] 
            
            _, selectedPs, leftRightIndexes = GetParcelNumbers(trackedArray, t) 
            zLevels = Z[selectedPs]

            mask_left = leftRightIndexes == -1
            mask_right = leftRightIndexes == 1
            
            for varName in varNames:
                masked_data = VARs[varName][selectedPs]
                
                NaN_mask = ~np.isnan(masked_data)
                masked_data_valid = masked_data[NaN_mask]
                zLevels_valid = zLevels[NaN_mask]
                
                # Get the pre-defined variable bins (e.g., 501 edges)
                v_bins = variableBinsDictionary[varName]
                
                # --- MAIN appending (All parcels) ---
                # Replaces: np.add.at(profileArray[varName]["profile_array"]...)
                BinVariable2D(profileArray[varName], zLevels_valid, masked_data_valid, z_bins, v_bins)
                
                # --- LEFT subset (-1 only) ---
                if np.any(mask_left):
                    mask_left_valid = mask_left[NaN_mask]
                    BinVariable2D(profileArray[varName], 
                                  zLevels_valid[mask_left_valid], 
                                  masked_data_valid[mask_left_valid], 
                                  z_bins, v_bins, suffix="_left")

                # --- RIGHT subset (+1 only) ---
                if np.any(mask_right):
                    mask_right_valid = mask_right[NaN_mask]
                    BinVariable2D(profileArray[varName], 
                                  zLevels_valid[mask_right_valid], 
                                  masked_data_valid[mask_right_valid], 
                                  z_bins, v_bins, suffix="_right")

    return profileArraysDictionary
    
def BinVariable2D(subDict, z_data, v_data, z_bins, v_bins, suffix=""):
    """
    Updates the 2D sum and count arrays within the dictionary.
    subDict: The dictionary at profileArray[varName]
    """
    # Matches your Initialization: "profile_array" and "profile_array_count"
    s_key = f"profile_array{suffix}"
    c_key = f"profile_array{suffix}_count"
    
    # Update Counts (2D Histogram)
    counts, _, _ = np.histogram2d(z_data, v_data, bins=[z_bins, v_bins])
    subDict[c_key] += counts.astype(np.int32)
    
    # Update Sums (Weighted 2D Histogram)
    sums, _, _ = np.histogram2d(z_data, v_data, bins=[z_bins, v_bins], weights=v_data)
    subDict[s_key] += sums

num_bins=500
def GetVariableBinsDictionary(num_bins=500): #*2dHistogram
    num_edges = num_bins + 1
    variableBinsDictionary = {}
    
    configs = {
        'W':           (-20, 50, "signed_log"),
        'CONVERGENCE': (-5e-4, 5e-4, "signed_log"),
        'QCQI':        (1e-6, 6e-3, "log"),
        'HMC':         (-5e-5, 5e-5, "signed_log"),
        # 'THETA_v':     (300, 350, "linear"), #all levels
        'THETA_v':     (305, 315, "linear"), #up to 3 km
        # 'THETA_e':     (320, 360, "linear"),  #all levels
        'THETA_e':     (330, 350, "linear"), #up to 3 km
        'MSE':         (3.3e5, 3.7e5, "linear"),
        'QV':          (0, 20/1e3, "linear"),
        'RH_vapor':    (0, 1.1, "linear"),
        'VMF_g':       (-20, 50, "linear"),
        'VMF_c':       (-50, 50, "linear")
    }


    for var, (mn, mx, btype) in configs.items():
        if btype == "linear":
            variableBinsDictionary[var] = np.linspace(mn, mx, num_edges)
            
        elif btype == "log":
            variableBinsDictionary[var] = np.logspace(np.log10(mn), np.log10(mx), num_edges)
            
        elif btype == "signed_log":
            # Determine threshold for the 'linear' center (e.g., 0.1)
            # Ensure we don't start logspace at 0 (log10(0) is undefined)
            threshold = 0.1 if var == 'W' else 1e-4
            
            # Positive side: threshold up to max
            pos = np.logspace(np.log10(threshold), np.log10(mx), num_bins // 2)
            
            # Negative side: min up to -threshold
            # We use abs(mn) to get the log range, then negate and reverse it
            neg = -np.logspace(np.log10(abs(mn)), np.log10(threshold), num_bins // 2)
            
            # neg is currently [-20, ..., -0.1]. These are already increasing.
            # pos is [0.1, ..., 50]. These are already increasing.
            # Combine: [Negatives] + [0] + [Positives]
            variableBinsDictionary[var] = np.concatenate([np.sort(neg), [0], np.sort(pos)])

    return variableBinsDictionary

In [ ]:
def RunCode():
    variableBinsDictionary = GetVariableBinsDictionary(num_bins=500) #*2dHistogram
    
    for t in tqdm(loop_elements, desc="Processing"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
        print("#" * 40,"\n",f"Processing timestep {t}/{loop_elements[-1]}")
        timeString = ModelData.timeStrings[t]
    
        #Forming Dictionary for Profile Arrays for current timestep
        trackedProfileArrays = CopyStructure(trackedArrays)
        profileArraysDictionary = InitializeProfileArrays(trackedProfileArrays,varNames)
        
        #getting variable data
        VARs = MakeDataDictionary(varNames, t)
        Z = GetSpatialData(t)
    
        #making tracked profiles
        profileArraysDictionary = MakeTrackedProfiles_2dHistogram(trackedArrays,profileArraysDictionary,varNames,VARs,Z,t,
                                                     variableBinsDictionary) #*2dHistogram
    
        #saving tracked profiles for current timestep
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, profileArraysDictionary, dataName, t)
    return profileArraysDictionary

In [ ]:
########################################
#RUNNING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                             Results_InputOutput_Class)
    
    #Removing After Ascent Count for SHALLOW parcels
    #Reason is there is a lot of shallow parcels
    #that hit their peak below 4 km, 
    #but stay in the cloud in turbulent eddies and later exit at much higher levels
    #and exit the cloudy updraft higher
    # for key1, subdict in trackedArrays.items():
    #     subdict['SHALLOW'][:,3]=0
    #testing without for now #*
    
    # Get Variable Names
    varNames = GetVarNames(dataName)

In [ ]:
if running:
    profileArraysDictionary = RunCode()

In [ ]:
#########################################
#RECOMBINE SEPERATE JOB_ARRAYS AFTER
recombine=False #KEEP FALSE WHEN JOBARRAY IS RUNNING
# recombine=True

In [ ]:
def RecombineProfiles(ModelData, DataManager, tIndicies, dataName):
    """
    Combine tracked profiles across specific time indices.
    Sums the 2D sum arrays and 2D count arrays.
    """
    print(f"Recombining {len(tIndicies)} TrackedProfiles files...\n")

    trackedProfileArrays = None

    for t in tqdm(tIndicies, desc="Combining Profiles", unit="timestep"):
        if "Entrainment" in dataName and t == ModelData.Ntime-1:
            continue
            
        profileArraysDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData, DataManager, dataName, t)

        if trackedProfileArrays is None:
            # Initialize with a deep copy of the first valid timestep
            trackedProfileArrays = copy.deepcopy(profileArraysDictionary)
            continue 

        # Sum the values from this timestep into the master dictionary
        for cat in profileArraysDictionary:        # e.g., 'CL'
            for depth in profileArraysDictionary[cat]:  # e.g., 'ALL'
                for var in profileArraysDictionary[cat][depth]: # e.g., 'W'
                    
                    # We loop through the standard keys you defined in initialization
                    # Note: We sum the entire 2D array (all heights and all bins)
                    target_keys = [
                        "profile_array",       "profile_array_count",
                        "profile_array_left",  "profile_array_left_count",
                        "profile_array_right", "profile_array_right_count"
                    ]
                    
                    for key in target_keys:
                        trackedProfileArrays[cat][depth][var][key] += \
                            profileArraysDictionary[cat][depth][var][key]

    return trackedProfileArrays

In [ ]:
if recombine==True:
    timeStrings_datetime = TrackedProfiles_DataLoading_CLASS.ConvertTimeStringsToDatetime(ModelData.timeStrings)
    timeSlice_all = np.arange(ModelData.Ntime)
    timeSlice_1 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,12) #late morning / pre-convective
    timeSlice_2 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,14) #CBL growth
    timeSlice_3 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,14,17) #mature convection / early decay

    ################################################################

    timeSlice_4 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,10,11) #SBF initiation 1
    timeSlice_5 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,11,12) #SBF initiation 2
    timeSlice_6 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,12,13) #CBL growth 1
    timeSlice_7 = TrackedProfiles_DataLoading_CLASS.FindTimeWindowIndices(timeStrings_datetime,13,14) #CBL growth 2


    timeSlices = [timeSlice_all,
                  timeSlice_1, timeSlice_2, timeSlice_3,
                  timeSlice_4, timeSlice_5, timeSlice_6, timeSlice_7]
    timeLabels = ['combined',
                  'combined_10-12', 'combined_12-14', 'combined_14-17']
                  # 'combined_10-11', 'combined_11-12', 'combined_12-13', 'combined_13-14']

In [ ]:
if recombine==True:
    # for dataName in ["Variables",
    #                  "Entrainment","PROCESSED_Entrainment","PROCESSED_Entrainment_DivideMassFlux",
    #                  "W_Budgets","QV_Budgets","TH_Budgets"]:
    for dataName in ["Variables"]:
        print(f"Working on {dataName}")

        for timeSlice, timeLabel in zip(timeSlices, timeLabels): #looping through above time windows
            trackedProfileArrays = RecombineProfiles(ModelData, DataManager_TrackedProfiles,tIndicies=timeSlice,dataName=dataName)
            TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, trackedProfileArrays, dataName, t=timeLabel)

In [ ]:
#########################
#PLOTTING FUNCTIONS
plotting=False
# plotting=True

In [ ]:
def PlotCFAD_Single(axis,
                   profile_array_count_raw, varName, parcelDepth, parcelType,
                   zTop_km=3): # Changed default to match your call
    # 1. Normalize frequency data
    c_data = profile_array_count_raw.astype(float)
    row_sums = c_data.sum(axis=1)
    
    # Avoid division by zero and normalize
    cfad_normalized = np.divide(c_data, row_sums[:, np.newaxis], 
                                where=row_sums[:, np.newaxis] != 0)
    
    # 2. Setup Bins and Edges
    variableBinsDictionary = GetVariableBinsDictionary(num_bins=500)
    x_edges = variableBinsDictionary[varName].copy() 
    if "Q" in varName: x_edges *= 1e3
    
    # Create y_edges (Height): length should be (number of rows + 1)
    z_centers = ModelData.zh
    dz = np.diff(z_centers)
    dz = np.append(dz, dz[-1]) 
    y_edges = np.append(z_centers - dz/2, z_centers[-1] + dz[-1]/2)

    # 3. Plotting
    cfad_masked = np.ma.masked_where(cfad_normalized <= 0, cfad_normalized)
    
    pcm = axis.pcolormesh(x_edges, y_edges, cfad_masked*100, 
                         shading='flat', 
                         cmap='turbo',
                         norm=mcolors.LogNorm(vmin=0.1, vmax=100))
    
    # Formatting
    # Use ax=axis for the colorbar so it doesn't steal space from the whole figure
    cb = plt.colorbar(pcm, ax=axis, label='Frequency (%)')
    axis.set_title(f'{parcelType} | {parcelDepth}')
    axis.set_xlabel(f'{varName} Value')
    axis.set_ylabel('Height (m MSL)')
    
    if varName.upper() in ['W', 'CONVERGENCE']:
        axis.set_xscale('symlog')
        axis.grid(True, which="both", ls="-", alpha=0.3)
    else:
        axis.grid(True, ls=":", alpha=0.5)
    
    # Note: If ModelData.zh is in meters, zTop_km might need to be zTop_km * 1000
    axis.set_ylim(0, zTop_km)

def CombinedPlot(varName='QV', parcelTypes=('CL','nonCL'), parcelDepths=('ALL',), sideOfSBF='_right'):
    # Determine grid dimensions
    ncols = len(parcelTypes)
    nrows = len(parcelDepths)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 3*nrows), squeeze=False)
    
    for r, pDepth in enumerate(parcelDepths):
        for c, pType in enumerate(parcelTypes):
            ax = axes[r, c]
            
            # Extract data using the loop variables
            data_key = f'profile_array{sideOfSBF}_count'
            profile_array_count = trackedProfileArrays[pType][pDepth][varName][data_key]
            
            # Call your plotting function
            PlotCFAD_Single(ax, profile_array_count, varName, pDepth, pType, zTop_km=3)
            
    plt.tight_layout()

# Example Call:
# CombinedPlot(varName='QV', parcelTypes=('CL','nonCL'), parcelDepths=('ALL', '0-500m'), sideOfSBF='_right')

# #improving bins
# a=trackedProfileArrays['CL']['ALL']['HMC']
# bins=variableBinsDictionary['HMC']
# bins[np.where(a['profile_array_count']!=0)[0]].min()

In [ ]:
#########################
#DATA LOADING

In [ ]:
if plotting:
    timeLabel='combined'
    trackedProfileArrays = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=timeLabel)

In [ ]:
#########################
#PLOTTING

In [ ]:
if plotting:
    CombinedPlot(varName='QV', parcelTypes=('CL','nonCL'), parcelDepths=('ALL','SHALLOW','DEEP'),sideOfSBF='_right')

In [ ]:
if plotting:
    CombinedPlot(varName='THETA_v', parcelTypes=('CL','nonCL'), parcelDepths=('ALL','SHALLOW','DEEP'),sideOfSBF='_right')

In [ ]:
if plotting:
    CombinedPlot(varName='THETA_e', parcelTypes=('CL','nonCL'), parcelDepths=('ALL','SHALLOW','DEEP'),sideOfSBF='_right')